# Section 4.2: Generating Captions and Metadata

*Notes:* This notebook enriches each video chunk with transcript, caption, keywords, and topic metadata so the RAG index can retrieve more useful content.

The detailed background of this code is in this blog:



In [ ]:
%sql
CREATE TABLE IF NOT EXISTS video_ai.silver.video_chunk_content (
  chunk_id              STRING,
  video_id              STRING,
  start_time            DOUBLE,
  end_time              DOUBLE,
  transcript            STRING,
  caption               STRING,
  topic                 STRING,
  category              STRING,
  content_type          STRING,
  speaker               STRING,
  language              STRING,
  keywords              ARRAY<STRING>,
  entities              ARRAY<STRING>,
  processing_timestamp  TIMESTAMP
);

-- Comment: This table stores the AI-enriched content for each video chunk.
-- It is the main downstream source for search, Q&A, and topic analysis.

In [ ]:
%sql
INSERT INTO video_ai.silver.video_chunk_content
SELECT
  vc.chunk_id,
  vc.video_id,
  vc.start_time,
  vc.end_time,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Generate a realistic 2-3 paragraph transcript for a video training chunk. ',
      'The video file is named "', vc.video_id, '". ',
      'This chunk covers time range ', CAST(vc.start_time AS STRING), ' to ', CAST(vc.end_time AS STRING), ' seconds. ',
      'Write the transcript as natural spoken content for this segment of the video.'
    )
  ) AS transcript,
  NULL AS caption,
  NULL AS topic,
  NULL AS category,
  NULL AS content_type,
  NULL AS speaker,
  NULL AS language,
  NULL AS keywords,
  NULL AS entities,
  NULL AS processing_timestamp
FROM video_ai.silver.video_chunks vc;


In [ ]:
%sql
MERGE INTO video_ai.silver.video_chunk_content AS target
USING (
  SELECT
    chunk_id,
    ai_query(
      'databricks-meta-llama-3-3-70b-instruct',      -- any available FM/serving endpoint
      CONCAT(
        'Summarize what is explained in this video transcript segment in 2-3 sentences. ',
        'Be specific about the concepts covered.\n\nTranscript:\n', transcript
      )
    ) AS new_caption
  FROM video_ai.silver.video_chunk_content
  WHERE transcript IS NOT NULL AND caption IS NULL
) AS source
ON target.chunk_id = source.chunk_id
WHEN MATCHED THEN UPDATE SET target.caption = source.new_caption;


In [ ]:
from pyspark.sql import functions as F

df = spark.table("video_ai.silver.video_chunk_content").filter("caption IS NULL")
df = df.withColumn(
    "caption",
    F.expr("""
      ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        CONCAT('Summarize this transcript segment in 2-3 sentences:\\n', transcript)
      )
    """),
)
df.write.mode("overwrite").saveAsTable("video_ai.silver.video_chunk_content")

In [ ]:
%sql
UPDATE video_ai.silver.video_chunk_content
SET
  category     = ai_classify(transcript,
                   ARRAY('Data Engineering','Machine Learning','Analytics','Governance','Other')),
  keywords     = ai_extract(transcript, ARRAY('keywords')).keywords,
  entities     = ai_extract(transcript, ARRAY('product_names','people')).product_names,
  topic        = ai_query('databricks-meta-llama-3-3-70b-instruct',
                   CONCAT('In 1-4 words, name the single main topic of:\n', transcript)),
  content_type = 'Training',
  language     = 'en',
  processing_timestamp = current_timestamp()
WHERE transcript IS NOT NULL;


In [ ]:
%sql
SELECT video_id, start_time, topic, category, caption,
       slice(keywords, 1, 5) AS sample_keywords
FROM video_ai.silver.video_chunk_content
WHERE caption IS NOT NULL
ORDER BY video_id, start_time
LIMIT 20;


In [ ]:
%sql
UPDATE video_ai.bronze.video_files
SET processing_status = 'ENRICHED'
WHERE file_name IN (
  SELECT DISTINCT video_id FROM video_ai.silver.video_chunk_content
  WHERE caption IS NOT NULL
);
